# Customer Clustering & Segmentation — Instacart Grocery Data
### Python · pandas · scikit-learn · Power BI

---

**Dataset:** Instacart Online Grocery Orders (real anonymised data, ~3.4M orders, 206K customers)  
**Source:** https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis

| Stage | Tool | Description |
|---|---|---|
| 1. Load & Inspect | pandas | Load all 6 Instacart files |
| 2. Clean & Join | pandas | Merge orders → products → departments |
| 3. Feature Engineering | pandas | Aggregate to one row per customer |
| 4. EDA | matplotlib / seaborn | Explore distributions and patterns |
| 5. Clustering | scikit-learn K-Means | Segment customers |
| 6. Profiling | pandas | Name and interpret each segment |
| 7. Visualisation | matplotlib / seaborn | Cluster profile charts |
| 8. Export | CSV | Power BI ready outputs |

---

### How to get the data
1. Go to https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis
2. Download and extract the zip
3. Place all 6 CSV files into a folder called `instacart_data/` next to this notebook

Files needed: `orders.csv`, `order_products__prior.csv`, `order_products__train.csv`, `products.csv`, `departments.csv`, `aisles.csv`

## 0. Install dependencies

In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn -q

## 1. Load & inspect raw data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
NAVY = '#1B2A4A'
TEAL = '#00838F'
BLUE = '#64B5F6'
RED  = '#EF9A9A'

DATA_DIR = 'instacart_data/'
OUT_DIR  = 'instacart_processed/'
os.makedirs(OUT_DIR, exist_ok=True)

orders      = pd.read_csv(f'{DATA_DIR}orders.csv')
order_prior = pd.read_csv(f'{DATA_DIR}order_products__prior.csv')
order_train = pd.read_csv(f'{DATA_DIR}order_products__train.csv')
products    = pd.read_csv(f'{DATA_DIR}products.csv')
departments = pd.read_csv(f'{DATA_DIR}departments.csv')
aisles      = pd.read_csv(f'{DATA_DIR}aisles.csv')

print('=== Dataset sizes ===')
for name, df in [('orders', orders), ('order_prior', order_prior), ('order_train', order_train),
                  ('products', products), ('departments', departments), ('aisles', aisles)]:
    print(f'  {name:<25} {len(df):>10,} rows  x  {df.shape[1]} cols')

In [ ]:
orders.head(3)

In [ ]:
order_prior.head(3)

In [ ]:
# Products enriched with department names
products_enriched = products.merge(departments, on='department_id').merge(aisles, on='aisle_id')
products_enriched[['product_id','product_name','department','aisle']].head(5)

## 2. Clean & join

In [ ]:
# Combine prior + train order-product lines
all_order_products = pd.concat([order_prior, order_train], ignore_index=True)
print(f'Total order-product lines: {len(all_order_products):,}')

# Join to orders to get user_id and order metadata
txn = (
    all_order_products
    .merge(orders[['order_id','user_id','order_number','order_dow',
                   'order_hour_of_day','days_since_prior_order']],
           on='order_id', how='inner')
    .merge(products_enriched[['product_id','product_name','department','aisle']],
           on='product_id', how='left')
)

# Basic cleaning
txn = txn.dropna(subset=['user_id'])
txn['reordered'] = txn['reordered'].fillna(0).astype(int)

print(f'Enriched transactions: {len(txn):,}')
print(f'Unique customers     : {txn["user_id"].nunique():,}')
txn.head(3)

## 3. Feature engineering
We aggregate to **one row per customer** — behavioural signals that describe how they shop.

In [ ]:
# Order-level aggregations first (then roll up to customer)
order_agg = (
    txn
    .groupby(['user_id', 'order_id'])
    .agg(
        basket_size     = ('product_id', 'count'),
        items_reordered = ('reordered', 'sum'),
    )
    .reset_index()
)

# Customer-level features
customer_features = (
    txn
    .groupby('user_id')
    .agg(
        total_orders            = ('order_id',              'nunique'),
        total_items_purchased   = ('product_id',            'count'),
        unique_products         = ('product_id',            'nunique'),
        unique_departments      = ('department',            'nunique'),
        reorder_rate            = ('reordered',             'mean'),
        avg_days_between_orders = ('days_since_prior_order','mean'),
        max_order_number        = ('order_number',          'max'),
    )
    .reset_index()
)

# Merge avg basket size from order-level
avg_basket = order_agg.groupby('user_id')['basket_size'].mean().reset_index()
avg_basket.columns = ['user_id', 'avg_basket_size']
customer_features = customer_features.merge(avg_basket, on='user_id')

# Top department per customer
top_dept = (
    txn
    .groupby(['user_id','department'])['product_id']
    .count()
    .reset_index(name='dept_count')
    .sort_values('dept_count', ascending=False)
    .drop_duplicates('user_id')
    [['user_id','department']]
    .rename(columns={'department':'top_department'})
)
customer_features = customer_features.merge(top_dept, on='user_id', how='left')

# Drop customers with no days_since_prior_order data (true one-time orderers)
customer_features = customer_features.dropna(subset=['avg_days_between_orders'])
customer_features = customer_features.round(3)

customer_features.to_csv(f'{OUT_DIR}customer_features.csv', index=False)
print(f'Customer feature table: {len(customer_features):,} customers x {len(customer_features.columns)} features')
customer_features.describe().round(2)

## 4. Exploratory Data Analysis

In [ ]:
# Feature distributions
num_features = ['total_orders','total_items_purchased','avg_basket_size',
                'avg_days_between_orders','reorder_rate','unique_departments','unique_products']
labels       = ['Total Orders','Total Items Purchased','Avg Basket Size',
                 'Avg Days Between Orders','Reorder Rate','Unique Departments','Unique Products']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Customer feature distributions — Instacart', fontsize=14, fontweight='bold', color=NAVY, y=1.01)

for ax, col, label in zip(axes.flatten(), num_features, labels):
    data = customer_features[col].dropna()
    ax.hist(data.clip(upper=data.quantile(0.99)), bins=40, color=TEAL, edgecolor='white', linewidth=0.4)
    ax.set_title(label, fontsize=10, fontweight='bold', color=NAVY)
    ax.set_ylabel('Customers', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes.flatten()[-1].set_visible(False)  # hide empty 8th panel
plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Shopping patterns — day of week + hour of day
dow_labels = ['Sunday','Monday','Tuesday','Wednesday','Thursday','Friday','Saturday']
order_dow  = orders['order_dow'].value_counts().sort_index()
order_hour = orders['order_hour_of_day'].value_counts().sort_index()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Shopping patterns', fontsize=13, fontweight='bold', color=NAVY)

ax1.bar(dow_labels, order_dow.values, color=TEAL, edgecolor='white')
ax1.set_title('Orders by day of week', fontweight='bold', color=NAVY)
ax1.set_ylabel('Number of orders')
ax1.tick_params(axis='x', rotation=30)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.plot(order_hour.index, order_hour.values, color=NAVY, linewidth=2.5, marker='o', markersize=4)
ax2.fill_between(order_hour.index, order_hour.values, alpha=0.12, color=TEAL)
ax2.set_title('Orders by hour of day', fontweight='bold', color=NAVY)
ax2.set_ylabel('Number of orders')
ax2.set_xlabel('Hour')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}shopping_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top departments by volume
dept_counts = (
    order_prior
    .merge(products[['product_id','department_id']], on='product_id')
    .merge(departments, on='department_id')
    .groupby('department')['product_id']
    .count()
    .sort_values(ascending=True)
    .tail(12)
)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(dept_counts.index, dept_counts.values, color=TEAL, edgecolor='white')
ax.set_title('Top departments by items ordered', fontsize=13, fontweight='bold', color=NAVY)
ax.set_xlabel('Items ordered')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
for bar, val in zip(bars, dept_counts.values):
    ax.text(val + 10000, bar.get_y() + bar.get_height()/2,
            f'{val/1e6:.1f}M', va='center', fontsize=9, color=NAVY)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}top_departments.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = customer_features[num_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.8},
            xticklabels=labels, yticklabels=labels)
ax.set_title('Feature correlation matrix', fontsize=13, fontweight='bold', color=NAVY, pad=12)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Preprocessing for clustering

In [ ]:
from sklearn.preprocessing import StandardScaler

CLUSTER_FEATURES = [
    'total_orders',
    'total_items_purchased',
    'avg_basket_size',
    'avg_days_between_orders',
    'reorder_rate',
    'unique_departments',
    'unique_products',
]

X = customer_features[CLUSTER_FEATURES].copy()

# Cap outliers at 99th percentile so extreme values don't distort clusters
for col in CLUSTER_FEATURES:
    X[col] = X[col].clip(upper=X[col].quantile(0.99))

# Standardise — brings all features to the same scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Feature matrix: {X_scaled.shape[0]:,} customers x {X_scaled.shape[1]} features')

## 6. K-Means clustering — finding optimal k

In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score

K_RANGE     = range(2, 9)
inertias    = []
silhouettes = []

# Silhouette on a sample — avoids slowness on 200K customers
SAMPLE_SIZE = 10000
sample_idx  = np.random.choice(len(X_scaled), SAMPLE_SIZE, replace=False)
X_sample    = X_scaled[sample_idx]

print('Testing k values...')
for k in K_RANGE:
    km     = MiniBatchKMeans(n_clusters=k, random_state=42, n_init=5, batch_size=5000)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_sample, km.predict(X_sample), random_state=42)
    silhouettes.append(sil)
    print(f'  k={k}  inertia={km.inertia_:,.0f}  silhouette={sil:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Choosing optimal number of clusters', fontsize=13, fontweight='bold', color=NAVY)

ax1.plot(list(K_RANGE), inertias, marker='o', color=TEAL, linewidth=2.5, markersize=8)
ax1.set_title('Elbow method — inertia', fontweight='bold', color=NAVY)
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

best_k_sil = list(K_RANGE)[np.argmax(silhouettes)]
ax2.plot(list(K_RANGE), silhouettes, marker='s', color=NAVY, linewidth=2.5, markersize=8)
ax2.axvline(x=best_k_sil, color=TEAL, linestyle='--', alpha=0.6, label=f'Best k={best_k_sil}')
ax2.set_title('Silhouette score (higher = better)', fontweight='bold', color=NAVY)
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Silhouette score')
ax2.legend()
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nSuggested optimal k: {best_k_sil}')

In [ ]:
# Fit final model — adjust OPTIMAL_K based on charts above
OPTIMAL_K = 4

final_km = MiniBatchKMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10, batch_size=5000)
customer_features['cluster'] = final_km.fit_predict(X_scaled)

print('Cluster distribution:')
print(customer_features['cluster'].value_counts().sort_index())

## 7. Cluster profiling & segment naming

In [ ]:
profile = (
    customer_features
    .groupby('cluster')[CLUSTER_FEATURES]
    .mean()
    .round(2)
)
profile['customer_count'] = customer_features['cluster'].value_counts().sort_index()
profile['pct_of_base']    = (profile['customer_count'] / profile['customer_count'].sum() * 100).round(1)

# Assign meaningful business names based on dominant signals
SEGMENT_NAMES = {
    profile['total_orders'].idxmax()            : 'Power Shoppers',
    profile['avg_days_between_orders'].idxmax() : 'Infrequent Buyers',
    profile['reorder_rate'].idxmax()            : 'Habitual Reorderers',
}
remaining = [c for c in range(OPTIMAL_K) if c not in SEGMENT_NAMES]
SEGMENT_NAMES[remaining[0]] = 'Casual Explorers'

customer_features['segment'] = customer_features['cluster'].map(SEGMENT_NAMES)
profile['segment'] = profile.index.map(SEGMENT_NAMES)

display_cols = ['segment','customer_count','pct_of_base','total_orders',
                'avg_basket_size','avg_days_between_orders','reorder_rate','unique_departments']
profile[display_cols]

## 8. Visualise cluster profiles

In [ ]:
SEGMENT_COLORS = {
    'Power Shoppers'      : '#1B2A4A',
    'Habitual Reorderers' : '#00838F',
    'Casual Explorers'    : '#64B5F6',
    'Infrequent Buyers'   : '#EF9A9A',
}
SEGMENT_ORDER = [s for s in SEGMENT_COLORS if s in customer_features['segment'].unique()]

seg_summary = (
    customer_features
    .groupby('segment')
    .agg(
        customer_count         = ('user_id',                'count'),
        avg_total_orders       = ('total_orders',           'mean'),
        avg_basket_size        = ('avg_basket_size',        'mean'),
        avg_days_between       = ('avg_days_between_orders','mean'),
        avg_reorder_rate       = ('reorder_rate',           'mean'),
        avg_unique_departments = ('unique_departments',     'mean'),
        avg_unique_products    = ('unique_products',        'mean'),
    )
    .round(2)
    .reindex(SEGMENT_ORDER)
    .reset_index()
)
colors = [SEGMENT_COLORS[s] for s in seg_summary['segment']]

def bar_chart(ax, col, title, fmt='{:.0f}'):
    bars = ax.bar(seg_summary['segment'], seg_summary[col],
                  color=colors, edgecolor='white', width=0.6)
    ax.set_title(title, fontsize=11, fontweight='bold', color=NAVY)
    ax.set_xticklabels(seg_summary['segment'], rotation=15, ha='right', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for bar, val in zip(bars, seg_summary[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                fmt.format(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Customer segment profiles — Instacart', fontsize=15, fontweight='bold', color=NAVY, y=1.01)

bar_chart(axes[0,0], 'customer_count',         'Segment size (customers)',       fmt='{:,.0f}')
bar_chart(axes[0,1], 'avg_total_orders',        'Avg total orders',               fmt='{:.1f}')
bar_chart(axes[0,2], 'avg_basket_size',         'Avg basket size (items)',        fmt='{:.1f}')
bar_chart(axes[1,0], 'avg_days_between',        'Avg days between orders',        fmt='{:.0f}d')
bar_chart(axes[1,1], 'avg_reorder_rate',        'Reorder rate',                   fmt='{:.2f}')
bar_chart(axes[1,2], 'avg_unique_departments',  'Avg unique departments shopped', fmt='{:.1f}')

plt.tight_layout()
plt.savefig(f'{OUT_DIR}cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Customer share pie chart
count_by_seg = seg_summary.set_index('segment')['customer_count']
seg_colors_ordered = [SEGMENT_COLORS[s] for s in count_by_seg.index]

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    count_by_seg.values,
    labels=count_by_seg.index,
    autopct='%1.1f%%',
    colors=seg_colors_ordered,
    startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2.5),
    pctdistance=0.75
)
for t in autotexts: t.set_fontsize(11); t.set_fontweight('bold'); t.set_color('white')
for t in texts: t.set_fontsize(11); t.set_color(NAVY)
ax.set_title('Customer share by segment', fontsize=14, fontweight='bold', color=NAVY, pad=20)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}segment_share.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top department per segment
top_dept_seg = (
    customer_features[['user_id','segment','top_department']]
    .groupby(['segment','top_department'])
    .size()
    .reset_index(name='count')
)
top_dept_seg['pct'] = top_dept_seg.groupby('segment')['count'].transform(lambda x: x/x.sum()*100)

fig, axes = plt.subplots(1, len(SEGMENT_ORDER), figsize=(18, 6))
fig.suptitle('Top departments by segment', fontsize=14, fontweight='bold', color=NAVY)

for ax, seg in zip(axes, SEGMENT_ORDER):
    data = top_dept_seg[top_dept_seg['segment'] == seg].nlargest(6, 'pct')
    ax.barh(data['top_department'], data['pct'],
            color=SEGMENT_COLORS[seg], edgecolor='white')
    ax.set_title(seg, fontsize=10, fontweight='bold', color=NAVY)
    ax.set_xlabel('% of customers')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.invert_yaxis()

plt.tight_layout()
plt.savefig(f'{OUT_DIR}dept_by_segment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Reorder rate distribution — box plot
fig, ax = plt.subplots(figsize=(12, 6))
data_to_plot = [customer_features[customer_features['segment'] == s]['reorder_rate'].values
                for s in SEGMENT_ORDER]
bp = ax.boxplot(data_to_plot, patch_artist=True, notch=False,
                medianprops=dict(color='white', linewidth=2.5))
for patch, seg in zip(bp['boxes'], SEGMENT_ORDER):
    patch.set_facecolor(SEGMENT_COLORS[seg])
    patch.set_alpha(0.85)
ax.set_xticklabels(SEGMENT_ORDER, fontsize=11)
ax.set_title('Reorder rate distribution by segment', fontsize=13, fontweight='bold', color=NAVY)
ax.set_ylabel('Reorder rate (0–1)', fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}reorder_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Commercial insights & recommendations

In [ ]:
summary_rows = []
for seg in SEGMENT_ORDER:
    d = customer_features[customer_features['segment'] == seg]
    summary_rows.append({
        'Segment'                 : seg,
        'Customers'               : f'{len(d):,}',
        '% of base'               : f'{len(d)/len(customer_features)*100:.1f}%',
        'Avg orders'              : f'{d["total_orders"].mean():.1f}',
        'Avg basket size'         : f'{d["avg_basket_size"].mean():.1f}',
        'Avg days between orders' : f'{d["avg_days_between_orders"].mean():.0f}',
        'Reorder rate'            : f'{d["reorder_rate"].mean():.2f}',
        'Avg departments'         : f'{d["unique_departments"].mean():.1f}',
    })

pd.DataFrame(summary_rows).set_index('Segment')

In [ ]:
print("""
══════════════════════════════════════════════════════════════
  COMMERCIAL RECOMMENDATIONS BY SEGMENT
══════════════════════════════════════════════════════════════

🏆 Power Shoppers
   → Highest frequency and basket size — highest value customers
   → Protect with loyalty rewards and early access to new products
   → Cross-sell into departments they haven't explored yet
   → Focus: retention and basket expansion

🔁 Habitual Reorderers
   → Strong brand loyalty — repeat the same products consistently
   → Push subscribe & save / auto-replenishment offers
   → Introduce adjacent products within their top departments
   → Focus: lock in the habit, then gradually expand their range

🛍️ Casual Explorers
   → Browse broadly but low reorder rate — still building habits
   → Curated bundles, seasonal promotions, personalised suggestions
   → Incentivise second and third orders to build frequency
   → Focus: convert casual trial into regular shopping habit

⏰ Infrequent Buyers
   → Long gaps between orders — at risk of lapsing
   → Time-sensitive win-back offers (free delivery, discount)
   → Identify what triggered their last order and replicate it
   → Focus: reduce time between orders, find reactivation triggers

══════════════════════════════════════════════════════════════
""")

## 10. Export for Power BI

In [ ]:
# Export 1: Customer-level with segment labels
customer_features.to_csv(f'{OUT_DIR}powerbi_customers.csv', index=False)

# Export 2: Segment summary KPIs
pd.DataFrame(summary_rows).to_csv(f'{OUT_DIR}powerbi_segment_summary.csv', index=False)

# Export 3: Orders with segment labels (for time-series views)
orders.merge(customer_features[['user_id','segment']], on='user_id', how='left') \
      .to_csv(f'{OUT_DIR}powerbi_orders.csv', index=False)

print('Exports saved to instacart_processed/')
print('  powerbi_customers.csv      — one row per customer with segment + all features')
print('  powerbi_segment_summary.csv — aggregate KPIs per segment')
print('  powerbi_orders.csv          — order-level with segment labels')
print(f'\nTotal customers segmented: {len(customer_features):,} across {OPTIMAL_K} segments')

## 11. Power BI dashboard guide

Load the 3 CSVs from `instacart_processed/` into Power BI Desktop.

### Relationship
```
powerbi_customers [user_id]  →  powerbi_orders [user_id]  (1:Many)
```

### Recommended pages

| Page | Visuals |
|---|---|
| **Executive summary** | KPI cards: Total Customers, Avg Orders, Avg Basket Size, Avg Reorder Rate · Donut: Customer share by segment |
| **Segment profiles** | Clustered bar: Avg Orders / Basket Size / Reorder Rate by segment · Scatter: Total Orders vs Avg Days Between Orders coloured by segment |
| **Shopping behaviour** | Bar: Avg Days Between Orders by segment · Bar: Avg Unique Departments · Slicer: segment filter |
| **Order trends** | Line: Orders over time by segment using `order_dow` and `order_hour_of_day` |

### DAX measures

```dax
Total Customers = DISTINCTCOUNT(powerbi_customers[user_id])

Avg Orders per Customer = AVERAGE(powerbi_customers[total_orders])

Avg Reorder Rate = AVERAGE(powerbi_customers[reorder_rate])

Avg Basket Size = AVERAGE(powerbi_customers[avg_basket_size])

Segment % =
    DIVIDE(
        COUNTROWS(powerbi_customers),
        CALCULATE(COUNTROWS(powerbi_customers), ALL(powerbi_customers[segment]))
    )

Power Shoppers Count =
    CALCULATE(
        COUNTROWS(powerbi_customers),
        powerbi_customers[segment] = "Power Shoppers"
    )
```